# Broad Earnings Event Study and PEAD

    Extends earnings analysis into a multi-name post-earnings drift workflow with event windows and sector splits.

    **Category:** Event studies

    **Primary API calls used in this candidate:**
    - `qj.fmp.get_earnings_calendar`
- `qj.fmp.get_earnings_surprises`
- `qj.eod.get_historical_prices`

    The notebook is self-contained: QuantJourney SDK calls fetch the data, while the research logic is calculated in pandas/numpy so the assumptions are visible and auditable.

## Preview Chart

![41_broad_event_study_pead](../outputs/candidates/41_broad_event_study_pead.png)

In [ ]:
import os
import math
import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from quantjourney.sdk import QuantJourneyAPI

qj = QuantJourneyAPI.from_env()

START = os.getenv("QJ_EXAMPLE_START", "2020-01-01")
END = os.getenv("QJ_EXAMPLE_END", "2026-06-06")

plt.style.use("default")
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


def unwrap(payload: Any) -> Any:
    """Return the useful data value from common QuantJourney response shapes."""
    if isinstance(payload, dict) and "data" in payload:
        payload = payload["data"]
    if isinstance(payload, dict) and "value" in payload:
        return payload["value"]
    return payload


def safe_call(label: str, fn, **kwargs) -> Any:
    """Run an SDK call and keep the notebook usable when an optional feed is unavailable."""
    try:
        out = fn(**kwargs)
        print(f"{label}: ok")
        return out
    except Exception as exc:
        print(f"{label}: unavailable ({type(exc).__name__}: {exc})")
        return None


def as_rows(payload: Any) -> list[dict[str, Any]]:
    value = unwrap(payload)
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        for key in ("rows", "data", "items", "prices", "results"):
            if isinstance(value.get(key), list):
                return value[key]
        return [value]
    return []


def price_frame(symbol: str, start: str = START, end: str = END) -> pd.DataFrame:
    payload = qj.eod.get_historical_prices(symbol=symbol, start_date=start, end_date=end)
    rows = as_rows(payload)
    if not rows and isinstance(unwrap(payload), dict):
        value = unwrap(payload)
        rows = value.get(symbol) or value.get(symbol.upper()) or []
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f"No price data returned for {symbol}")
    df["date"] = pd.to_datetime(df["date"])
    for col in ["open", "high", "low", "close", "adjusted_close", "volume"]:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    if "adjusted_close" in df and df["adjusted_close"].notna().any():
        df["price"] = df["adjusted_close"].fillna(df["close"])
    else:
        df["price"] = df["close"]
    if "volume" not in df:
        df["volume"] = np.nan
    return df.dropna(subset=["price"]).sort_values("date").set_index("date")


def price_panel(symbols: list[str], start: str = START, end: str = END) -> tuple[pd.DataFrame, pd.DataFrame]:
    prices = {}
    volumes = {}
    for symbol in symbols:
        df = price_frame(symbol, start=start, end=end)
        prices[symbol] = df["price"]
        volumes[symbol] = df["volume"]
    return pd.DataFrame(prices).dropna(how="all"), pd.DataFrame(volumes).reindex(pd.DataFrame(prices).index)


def returns(prices: pd.DataFrame) -> pd.DataFrame:
    return prices.pct_change().replace([np.inf, -np.inf], np.nan).dropna(how="all")


def max_drawdown(nav: pd.Series) -> float:
    drawdown = nav / nav.cummax() - 1
    return float(drawdown.min())


def performance_stats(ret: pd.Series) -> pd.Series:
    ret = ret.dropna()
    nav = (1 + ret).cumprod()
    ann_ret = nav.iloc[-1] ** (252 / len(ret)) - 1 if len(ret) and nav.iloc[-1] > 0 else np.nan
    ann_vol = ret.std() * np.sqrt(252)
    sharpe = ann_ret / ann_vol if ann_vol and np.isfinite(ann_vol) else np.nan
    return pd.Series({
        "annual_return": ann_ret,
        "annual_volatility": ann_vol,
        "sharpe": sharpe,
        "max_drawdown": max_drawdown(nav) if len(nav) else np.nan,
        "total_return": nav.iloc[-1] - 1 if len(nav) else np.nan,
    })


def inverse_vol_weights(ret: pd.DataFrame) -> pd.Series:
    vol = ret.std().replace(0, np.nan)
    inv = 1 / vol
    return (inv / inv.sum()).fillna(0)


def min_variance_weights(ret: pd.DataFrame, ridge: float = 1e-4) -> pd.Series:
    cov = ret.cov().fillna(0).to_numpy() * 252
    cov = cov + np.eye(cov.shape[0]) * ridge
    inv = np.linalg.pinv(cov)
    raw = inv @ np.ones(cov.shape[0])
    raw = np.maximum(raw, 0)
    if raw.sum() == 0:
        raw = np.ones(cov.shape[0])
    return pd.Series(raw / raw.sum(), index=ret.columns)


def portfolio_returns(ret: pd.DataFrame, weights: pd.Series) -> pd.Series:
    aligned = ret[weights.index].dropna()
    return aligned @ weights.reindex(aligned.columns).fillna(0)


def risk_contribution(ret: pd.DataFrame, weights: pd.Series) -> pd.Series:
    aligned = ret[weights.index].dropna()
    cov = aligned.cov() * 252
    w = weights.reindex(cov.columns).fillna(0).to_numpy()
    port_var = float(w @ cov.to_numpy() @ w)
    if port_var <= 0:
        return pd.Series(0.0, index=cov.columns)
    contrib = w * (cov.to_numpy() @ w) / port_var
    return pd.Series(contrib, index=cov.columns)


def rolling_betas(y: pd.Series, x: pd.DataFrame, window: int = 126) -> pd.DataFrame:
    data = pd.concat([y.rename("asset"), x], axis=1).dropna()
    rows = []
    for i in range(window, len(data)):
        chunk = data.iloc[i - window:i]
        yy = chunk["asset"].to_numpy()
        xx = np.column_stack([np.ones(len(chunk)), chunk[x.columns].to_numpy()])
        beta = np.linalg.lstsq(xx, yy, rcond=None)[0][1:]
        rows.append(dict(date=data.index[i], **{col: beta[j] for j, col in enumerate(x.columns)}))
    return pd.DataFrame(rows).set_index("date") if rows else pd.DataFrame(columns=x.columns)


def zscore(s: pd.Series, window: int = 252) -> pd.Series:
    return (s - s.rolling(window).mean()) / s.rolling(window).std()


def dollar_adv(prices: pd.DataFrame, volumes: pd.DataFrame, window: int = 63) -> pd.DataFrame:
    return (prices * volumes).rolling(window).mean()


def plot_nav(ret_map: dict[str, pd.Series], title: str) -> None:
    fig, ax = plt.subplots()
    for label, ret in ret_map.items():
        nav = (1 + ret.dropna()).cumprod()
        ax.plot(nav.index, nav, label=label)
    ax.set_title(title)
    ax.legend()
    plt.show()


In [ ]:
symbols = ["AAPL", "MSFT", "NVDA", "GOOGL", "AMZN", "META"]
calendar = safe_call("FMP earnings calendar", qj.fmp.get_earnings_calendar, from_date="2024-01-01", to_date=END)
surprises = {symbol: safe_call(f"FMP earnings surprises {symbol}", qj.fmp.get_earnings_surprises, symbol=symbol) for symbol in symbols}
prices, volumes = price_panel(symbols, start="2023-01-01", end=END)


In [ ]:
event_rows = []
for symbol, payload in surprises.items():
    for item in as_rows(payload):
        event_date = pd.to_datetime(item.get("date") or item.get("fiscalDateEnding"), errors="coerce")
        surprise = pd.to_numeric(item.get("surprisePercentage") or item.get("surprise"), errors="coerce")
        if pd.notna(event_date):
            event_rows.append({"symbol": symbol, "event_date": event_date, "surprise": surprise})
events = pd.DataFrame(event_rows)
if events.empty:
    events = pd.DataFrame({"symbol": symbols, "event_date": [prices.index[-126]] * len(symbols), "surprise": np.nan})
curves = []
for row in events.dropna(subset=["event_date"]).itertuples():
    if row.symbol not in prices:
        continue
    idx = prices.index.searchsorted(row.event_date)
    if idx < 21 or idx + 42 >= len(prices):
        continue
    window = prices[row.symbol].iloc[idx - 10:idx + 43]
    curve = window / window.iloc[10] - 1
    curves.append(pd.Series(curve.values, index=range(-10, len(curve) - 10), name=row.symbol))
event_curve = pd.concat(curves, axis=1) if curves else pd.DataFrame()
display(events.head())
if not event_curve.empty:
    event_curve.mean(axis=1).plot(title="Average post-earnings drift curve")
    plt.axvline(0, color="black", linestyle="--")
    plt.ylabel("Return vs event day")
    plt.show()


## Notes

This is a candidate workflow. In production, tenant scopes, connector allowlists,
provider metadata, request IDs and audit logs should be retained next to the resulting
tables or charts.